In [1]:
import pandas as pd

from evaluation.config import PROJECT_DIR
from granuscore.loader import JSONLineReader

reader = JSONLineReader()

In [2]:
def abbreviate(name):
    repl = {
        "scope_": "",
        "document": "doc",
        "sentence": "sent",
        "spooling_": "",
        "pooling": "pool",
        "doc_mean_": "doc_",
        "lower_quantile_mean": "lqm",
        "stailq": "stq",
        "tailq_": "",
    }

    for k, v in repl.items():
        name = name.replace(k, v)

    name = name.replace("_", "-")
    if "stq-" in name:
        stq = name.split("stq-")[1].split("-")[0]
        name = name.replace("sent-lqm-", f"sent-lqm-{stq}-")
        name = name.replace(f"-stq-{stq}", "")
    return name

stats = []

data = reader.read(f'{PROJECT_DIR}/data/s2orc/evaluations.jsonl')

for d in data:
    name = d['name']
    stats.append({
        'name': abbreviate(name),
        'correct_ordering': d['correct_ordering'],
    })
stats = pd.DataFrame(stats)

In [3]:
stats

,name,correct_ordering
0,sent-weighted-mean-pool-sum,0.584867
1,sent-weighted-mean-pool-mean,0.664622
2,sent-weighted-mean-pool-lqm-0.1,0.626789
3,sent-weighted-mean-pool-lqm-0.3,0.668712
4,sent-weighted-mean-pool-lqm-0.5,0.672802
...,...,...
58,doc-pool-lqm-0.1,0.648262
59,doc-pool-lqm-0.3,0.656442
60,doc-pool-lqm-0.5,0.660532
61,doc-pool-min,0.599182


In [4]:
main_stats = stats.copy().rename(columns={
    "name": "Method",
    "correct_ordering": "Ordering Acc.",
})

num_cols = main_stats.columns.drop("Method")
main_stats[num_cols] = (main_stats[num_cols] * 100).round(2)

# copy for formatting
main_stats_fmt = main_stats.copy()

for col in num_cols:
    # get indices of largest and second largest
    order = main_stats[col].sort_values(ascending=False).index
    best = order[0]
    second = order[1]

    for i in stats.index:
        val = main_stats.loc[i, col]

        if i == best:
            main_stats_fmt.loc[i, col] = f"\\textbf{{{val:.2f}}}"
        elif i == second:
            main_stats_fmt.loc[i, col] = f"\\textit{{{val:.2f}}}"
        else:
            main_stats_fmt.loc[i, col] = f"{val:.2f}"

# export latex
latex = main_stats_fmt.to_latex(index=False, escape=False)

print(latex)

\begin{tabular}{ll}
\toprule
Method & Ordering Acc. \\
\midrule
sent-weighted-mean-pool-sum & 58.49 \\
sent-weighted-mean-pool-mean & 66.46 \\
sent-weighted-mean-pool-lqm-0.1 & 62.68 \\
sent-weighted-mean-pool-lqm-0.3 & 66.87 \\
sent-weighted-mean-pool-lqm-0.5 & 67.28 \\
sent-weighted-mean-pool-min & 61.55 \\
sent-weighted-mean-pool-max & 58.79 \\
sent-sum-pool-sum & 47.55 \\
sent-sum-pool-mean & 45.19 \\
sent-sum-pool-lqm-0.1 & 48.67 \\
sent-sum-pool-lqm-0.3 & 47.03 \\
sent-sum-pool-lqm-0.5 & 45.71 \\
sent-sum-pool-min & 48.67 \\
sent-sum-pool-max & 44.38 \\
sent-mean-pool-sum & 60.53 \\
sent-mean-pool-mean & 67.48 \\
sent-mean-pool-lqm-0.1 & 62.88 \\
sent-mean-pool-lqm-0.3 & 66.05 \\
sent-mean-pool-lqm-0.5 & 66.16 \\
sent-mean-pool-min & 62.07 \\
sent-mean-pool-max & 59.10 \\
sent-lqm-0.9-pool-sum & 60.84 \\
sent-lqm-0.8-pool-sum & 62.27 \\
sent-lqm-0.7-pool-sum & 62.68 \\
sent-lqm-0.9-pool-mean & \textit{68.10} \\
sent-lqm-0.8-pool-mean & \textbf{68.71} \\
sent-lqm-0.7-pool-mean & 6

/var/folders/pf/gbn3rrqn3sn7212qgqt0n3rw0000gn/T/ipykernel_99273/3509615218.py:26: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '58.49' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  main_stats_fmt.loc[i, col] = f"{val:.2f}"


In [5]:
import numpy as np
import pandas as pd

df = main_stats_fmt.copy()
df = df[["Method", "Ordering Acc."]].reset_index(drop=True)

n_blocks = 3  # number of method/score pairs per row

blocks = np.array_split(df, n_blocks)

# reset index in each block so concat aligns row-wise
blocks = [b.reset_index(drop=True) for b in blocks]

# rename columns so headers repeat nicely
blocks = [
    b.rename(columns={"Method": "Method", "Ordering Acc.": "Ord. Acc."})
    for b in blocks
]

table_df = pd.concat(blocks, axis=1)

latex = table_df.to_latex(index=False, escape=False)
print(latex)

\begin{tabular}{llllll}
\toprule
Method & Ord. Acc. & Method & Ord. Acc. & Method & Ord. Acc. \\
\midrule
sent-weighted-mean-pool-sum & 58.49 & sent-lqm-0.9-pool-sum & 60.84 & sent-min-pool-sum & 60.02 \\
sent-weighted-mean-pool-mean & 66.46 & sent-lqm-0.8-pool-sum & 62.27 & sent-min-pool-mean & 66.16 \\
sent-weighted-mean-pool-lqm-0.1 & 62.68 & sent-lqm-0.7-pool-sum & 62.68 & sent-min-pool-lqm-0.1 & 61.76 \\
sent-weighted-mean-pool-lqm-0.3 & 66.87 & sent-lqm-0.9-pool-mean & \textit{68.10} & sent-min-pool-lqm-0.3 & 64.01 \\
sent-weighted-mean-pool-lqm-0.5 & 67.28 & sent-lqm-0.8-pool-mean & \textbf{68.71} & sent-min-pool-lqm-0.5 & 65.03 \\
sent-weighted-mean-pool-min & 61.55 & sent-lqm-0.7-pool-mean & 67.59 & sent-min-pool-min & 59.82 \\
sent-weighted-mean-pool-max & 58.79 & sent-lqm-0.9-pool-lqm-0.1 & 63.80 & sent-min-pool-max & 60.53 \\
sent-sum-pool-sum & 47.55 & sent-lqm-0.9-pool-lqm-0.3 & 66.16 & sent-max-pool-sum & 52.66 \\
sent-sum-pool-mean & 45.19 & sent-lqm-0.9-pool-lqm-0.5 & 

/Users/lukasellinger/PycharmProjects/granuscore/.venv/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
